In [ ]:
import sys
from pathlib import Path

import astropy.constants as apconst
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Add parent directory to path so we can import the modules
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dust_jwst_highz.data import draine03_dust_model, hirashita19_attenuation_curve
from dust_jwst_highz.model.cosmology import cosmo
from dust_jwst_highz.model.dust import (
    compute_mdust_steps,
    normed_number_weighted_grain_dist,
    optical_depth,
    transmission_sphere_mixed,
)
from dust_jwst_highz.model.ism import density_compression_ratio, lognormal_variance_from_mach, sample_surface_density
from dust_jwst_highz.model.luminosity import compute_l1500_steps, l1500_to_muv_conv
from dust_jwst_highz.model.star_formation import halo_to_stellar_mass, star_formation_history

DATA_DIR = Path("../data")

lambda_uv = 1500
kv = 8.55e3 / (1 - 0.674)  # cm^2/g
kuv = 3.92e4 / (1 - 0.3807)  # cm^2/g  # 1400 angstrom
kuv_hir = kv * hirashita19_attenuation_curve()(lambda_uv)
kuv_drn = kv * draine03_dust_model()(lambda_uv)

londa = np.array([
    4217., 3981., 3758., 3548., 3350., 3162., 2985., 2818., 2661., 2512.,
    2371., 2239., 2113., 1995., 1884., 1778., 1679., 1585., 1496., 1413.,
    1334., 1259., 1189., 1122., 1059., 1000., 944., 900.
]) / 1600.0  # adimensional # fmt: skip

# Grain size bins [cm], logarithmic in microns
a = np.array([
    1.000e-03, 1.5849e-03, 2.5119e-03, 3.9811e-03, 6.3096e-03, 1.0000e-02,
    1.5849e-02, 2.5119e-02, 3.9811e-02, 6.3096e-02, 1.0000e-01, 1.5849e-01,
    2.5119e-01, 3.9811e-01, 6.3096e-01, 1.0000e+00
]) * 1e-4  # convert microns to cm # fmt: skip

# Q_abs: wavelength and grain-size-dependent absorption efficiency
# Shape should be [len(wavelengths) * (len(a) - 1)], flattened
Qabs = np.loadtxt(DATA_DIR / "Q_abs_Draine_silicates.txt", unpack=True, delimiter=",")

# ======== INPUT DATA ===========
# Load SB99 instantaneous burst tables
time_yr, log_snr_yr = np.loadtxt(DATA_DIR / "snr_inst_Z001.txt", usecols=(0, 1), unpack=True)
time_yr_l1500, l1500_sb99 = np.loadtxt(DATA_DIR / "L1500_inst_Z001.txt", usecols=(0, 1), unpack=True)

# Load Hirashita attenuation curve (convert 1/micron to Angstrom)
inv_micron, Alambda_over_Av_hir = np.loadtxt(DATA_DIR / "Hirashita_dense_01Gyr_AttCurve.txt", unpack=True)
lambda_ang_hir = 1e4 / inv_micron

# Load Draine dust model optical properties
lambda_ang_drn = np.loadtxt(DATA_DIR / "Draine_MWDustRv31_Optical_prop.txt", usecols=0) * 1e4  # Convert to Angstrom

# Load Hirashita data
hir_a, hir_a4n = np.loadtxt(DATA_DIR / "Hirashita_a4n_dense_01Gyr.txt", unpack=True, delimiter=",")
hir_a_cm = hir_a * 1e-4  # convert micron to cm

redshift = 10.0
log_mh = 10.9
fb = cosmo.Ob(redshift) / cosmo.Om(redshift)
epsilon = 0.1
yd = 0.1

# Stellar mass from halo mass
Mstar_array = halo_to_stellar_mass(10**log_mh, fb, epsilon)

# Build SFH
tstep = 1  # [Myr]
len_sp_dis = 1000
spin_param_distr = np.random.lognormal(mean=np.log(10**-1.5677), sigma=0.5390, size=len_sp_dis)
sfh, log_mst_build, age = star_formation_history(10**log_mh, redshift, tstep, epsilon)

N_SN_arr, Md_arr = compute_mdust_steps(age, tstep, sfh, time_yr, log_snr_yr, yd)

# Compute 1500A luminosity [erg/s/Hz]
L1500_arr = compute_l1500_steps(l1500_sb99, age, tstep, sfh, time_yr_l1500, method="SB99")
Lintr = L1500_arr[-1]  # only last step (final galaxy age) # erg/s/Hz
MUV_intr = l1500_to_muv_conv(Lintr.value)
L1500_lambda = Lintr * apconst.c.value / (1500.0**2 * 1e-8)  # erg/s/Å
print("MUV (intrinsic) -->", MUV_intr)

In [ ]:
## Constants for the dust model (Draine+03 or Hirashita+19)
print("\n\nkuv/kV (tabulated input Draine):", kuv / kv)
print("kuv from Hirashita:", kuv_hir)
print("kuv from Draine model:", kuv_drn)
print("kuv/kV (Hirashita):", kuv_hir / kv)
print("kuv/kV (interp. Draine):", kuv_drn / kv)

# Plot attenuation curves in wavelength space
fig, ax = plt.subplots(figsize=(5, 5))

# Plot attenuation curves
ax.plot(lambda_ang_hir, Alambda_over_Av_hir, lw=2, alpha=0.8, label="Hirashita+19, (0.1-0.3) Gyr", color="teal")
ax.plot(
    lambda_ang_drn, draine03_dust_model()(lambda_ang_drn), lw=2.5, alpha=0.5, color="crimson", label="Draine+03, MW"
)

# Mark UV wavelength
ax.axvline(lambda_uv, color="gray", linestyle="--", lw=1)
ax.text(1550, 0, r"$\lambda=1500\ \mathrm{[\dot{A}]}$", fontsize=16, color="gray")

# Plot k_UV/k_V ratios
ax.scatter(
    lambda_uv, kuv_drn / kv, label="$k_{1500}/k_V$", color="crimson", marker="D", s=60, edgecolor="black", alpha=0.4
)
ax.scatter(lambda_uv, kuv_hir / kv, color="teal", marker="D", s=60, edgecolor="black", alpha=0.7)

# Configure axes
ax.set_xscale("log")
ax.set_xticks([1500, 2175, 3543, 4770, 6231, 7625, 9134])
ax.set_xlim(1e3, 2e4)
ax.set_ylim(-0.2, 5)
ax.set_xlabel(r"$\lambda\ \mathrm{[\dot{A}]}$", fontsize=20)
ax.set_ylabel(r"$\tau_{\lambda} / \tau_V$", fontsize=20)
ax.grid(True, alpha=0.2, lw=0.5)
ax.legend(frameon=False, fontsize=14, loc="upper left")

# Inset: Grain size distribution comparison
inset_ax = ax.inset_axes([0.55, 0.5, 0.4, 0.4])

# Compute Draine+03 grain size distribution (a^4 * n(a))
a_cm = np.linspace(1e-8, 1e-4, 10000)
a_mids = 0.5 * (a_cm[:-1] + a_cm[1:])
draine_a4n = a_mids**4 * normed_number_weighted_grain_dist(a_cm) / np.diff(a_cm)

# Plot grain size distributions
inset_ax.plot(hir_a, hir_a4n / np.max(hir_a4n), label="Hirashita+19", color="teal", alpha=0.8)
inset_ax.plot(1e4 * a_mids, draine_a4n / np.max(draine_a4n), label="Draine+03 - MW", color="crimson", alpha=0.5)
inset_ax.plot(1e4 * a_mids, a_mids**0.5 / np.max(a_mids**0.5), lw=0.8, ls=":", color="crimson", alpha=0.5, label="MRN")

# Configure inset axes
inset_ax.set_xscale("log")
inset_ax.set_yscale("log")
inset_ax.set_xlabel(r"Grain size $a$ [$\mu$m]", fontsize=16)
inset_ax.set_ylabel("$a^4 n(a)$ (normalized)", fontsize=16)
inset_ax.set_xlim(1e-3, 1)
inset_ax.set_ylim(1e-3, 2)
inset_ax.grid(True, which="both", ls=":", alpha=0.1)

plt.tight_layout()
plt.show()

In [ ]:
# ======== BUILD DUST SURFACE DENSITY DISTRIBUTION ===========
### === Setting up shell model with log-normal surface density fluctuations ===


# Fixed total dust mass [Msun]
M_dust = Md_arr[-1]
print("log Mdust/Msun -->", np.log10(M_dust))
print("log Mstar/Msun -->", np.log10(Mstar_array))

# Median sigma_d for uniform distribution [g/cm^2]
tauuv_arr = optical_depth(kuv, M_dust, 10**log_mh, spin_param_distr, redshift)
Sigmad_arr = tauuv_arr / kuv  # g/cm^2

# Log-normal draw for sigma_d, width set by Mach number
# Mach=3000  # turbulent Mach number
Sigmad_distr_10 = sample_surface_density(mu_sigma=np.median(Sigmad_arr), mach=10)
Sigmad_distr_300 = sample_surface_density(mu_sigma=np.median(Sigmad_arr), mach=300)

# Build the y-grid (dex) covering your histogram range
R_10 = density_compression_ratio(10)
sigma_ln_sigma_sq_10 = np.log(1 + (R_10 * 10**2) / 4)
sigma_ln_10 = np.sqrt(sigma_ln_sigma_sq_10)
mu_ln_10 = np.log(np.median(Sigmad_arr))
y_vals = np.linspace(-10, -2, 1000)  # dex
x_vals = 10**y_vals

# Transform to the PDF in y = log10 x (units: 1/dex)
pdf_y_10 = np.log(10) * x_vals * stats.lognorm.pdf(x_vals, s=sigma_ln_10, scale=np.exp(mu_ln_10))
# Overlay the correctly transformed PDF (no arbitrary /max)
# plt.plot(y_vals, pdf_y_10, label='PDF Mach=10 (correct in log10-space)', lw=2)

# --- Plot Sigma_dust distribution
plt.hist(
    np.log10(Sigmad_distr_10),
    bins=30,
    alpha=0.6,
    histtype="step",
    color="mistyrose",
    edgecolor="firebrick",
    label="Turbulence-driven distribution (Mach=10)",
    ls="--",
    lw=1.5,
    density=True,
)
plt.hist(
    np.log10(Sigmad_distr_300),
    bins=30,
    alpha=0.3,
    histtype="step",
    color="midnightblue",
    edgecolor="midnightblue",
    label="Turbulence-driven distribution (Mach=300)",
    ls="--",
    lw=1.0,
    density=True,
)
plt.hist(
    np.log10(Sigmad_arr),
    bins=30,
    alpha=0.2,
    lw=2.5,
    histtype="stepfilled",
    color="grey",
    edgecolor="dimgrey",
    label="Spin-driven distribution",
    density=True,
)

plt.axvline(np.log10(np.percentile(Sigmad_distr_10, 16)), color="firebrick", linestyle="--", lw=2.0, alpha=0.6)
plt.axvline(np.log10(np.percentile(Sigmad_distr_10, 84)), color="firebrick", linestyle="--", lw=2.0, alpha=0.6)

plt.axvline(np.log10(np.percentile(Sigmad_distr_300, 16)), color="midnightblue", linestyle="--", alpha=0.4)
plt.axvline(np.log10(np.percentile(Sigmad_distr_300, 84)), color="midnightblue", linestyle="--", alpha=0.4)

plt.axvline(np.log10(np.percentile(Sigmad_arr, 16)), color="dimgrey", lw=2.5, alpha=0.4)
plt.axvline(np.log10(np.percentile(Sigmad_arr, 84)), color="dimgrey", lw=2.5, alpha=0.4)
plt.xlabel(r"$\log_{10}(\Sigma_{d}/M_{\odot}\,\mathrm{kpc}^{-2})$")
plt.ylabel("PDF")
plt.legend(frameon=False, fontsize=14, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
albedo = 0.3807
Mach_array = np.array([5, 10, 20, 30, 40, 50, 100, 200, 300])

LUV_mean_vals, LIR_vals = [], []

# --- Loop over Mach numbers ---
plt.figure(figsize=(7, 6))
for i, Mach in enumerate(Mach_array):
    # --- parameters of Σ_d lognormal (in linear space) ---
    mu_ln = np.log(np.median(Sigmad_arr))
    sigma_ln = lognormal_variance_from_mach(Mach)

    # === UV: keep the Monte-Carlo distribution (NO PDF integration) ===
    Sigmad_distr = sample_surface_density(mu_sigma=np.median(Sigmad_arr), mach=Mach)  # linear Σ_d
    tauuv_distr = kuv * Sigmad_distr
    T_uv_draws = transmission_sphere_mixed(tauuv_distr)  # per-LOS transmission
    L1500_distr = Lintr * T_uv_draws  # per-LOS UV luminosities
    L1500_mean = np.median(L1500_distr)  # optional sample mean (not a PDF integral)
    LUV_mean_vals.append(L1500_mean)

    # plot the distribution (spread) and the sample mean for UV
    plt.scatter(
        Mach * np.ones(len(L1500_distr)), L1500_distr, facecolor="royalblue", edgecolor="darkblue", alpha=0.25, s=8
    )
    plt.scatter(
        Mach,
        L1500_mean,
        facecolor="royalblue",
        edgecolor="darkblue",
        alpha=0.5,
        label="$L_{\\rm UV}$ (diff. LOS)" if i == 0 else None,
    )

    # === IR: integrate against the continuous PDF ===
    # build an x-grid that captures essentially all probability
    x_min = np.exp(mu_ln - 6 * sigma_ln)
    x_max = np.exp(mu_ln + 6 * sigma_ln)
    x = np.logspace(np.log10(x_min), np.log10(x_max), 2000)
    p_x = stats.lognorm.pdf(x, s=sigma_ln, scale=np.exp(mu_ln))  # ~normalized over [x_min, x_max]

    tau_abs = kuv * x * (1.0 - albedo)  # absorption optical depth
    T_abs = transmission_sphere_mixed(tau_abs)

    LIR_total = Lintr.value * np.trapezoid((1.0 - T_abs) * p_x, x)  # E[absorbed UV]
    LIR_vals.append(LIR_total)

    plt.scatter(
        Mach,
        LIR_total,
        facecolor="coral",
        edgecolor="darkred",
        alpha=0.9,
        label="$L_{\\rm IR}$ (PDF-integrated)" if i == 0 else None,
    )

# -- Uniform case (unchanged logic) --
T_1500_uniform = transmission_sphere_mixed(tauuv_arr)
L1500_uniform = Lintr.value * T_1500_uniform
L1500_absorbed_uniform = Lintr.value * (1 - T_1500_uniform) * (1 - albedo)
LIR_uniform = L1500_absorbed_uniform
L1500_uniform_mean = np.median(L1500_uniform)
LIR_uniform_mean = np.median(LIR_uniform)
print(
    f"Uniform case --> Luv={L1500_uniform_mean:.2e}, Lir={LIR_uniform_mean:.2e}, "
    f"Lir/Luv={LIR_uniform_mean / L1500_uniform_mean:.2f}"
)

# -- Finalize plot --
plt.axhline(Lintr.value, color="black", label="$L_{\\rm intr}$", lw=2, alpha=0.5)
plt.axhline(
    np.median(L1500_uniform), color="royalblue", linestyle="--", label="$L_{\\rm UV}$ (uniform)", lw=2, alpha=0.5
)
plt.axhline(np.median(LIR_uniform), color="coral", linestyle="--", label="$L_{\\rm IR}$ (uniform)", lw=2, alpha=0.5)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"Mach number $\mathcal{M}$ ")
plt.ylabel("Luminosity [erg/s/Å]")
plt.legend(frameon=False, fontsize=14)
plt.tight_layout()
plt.show()